# Notebook 04 — Link RxHandBD Prescription Images to MPI

**Goal:** Assign each prescription image a `patient_id` from the MPI.

**Linking logic:**
- Fuzzy match each image's medicine label against patient's `primary_medication`
- If match score >= 70 → assign matched patient
- If no match found → random patient assignment

**Output:** `data_preparation/linked/prescriptions_linked.csv`

## 1. Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import random
import os
from fuzzywuzzy import fuzz

RXHANDBD_DIR    = "../raw/rxhandbd/"
TRAIN_LABELS    = os.path.join(RXHANDBD_DIR, "Train_Label.csv")
TEST_LABELS     = os.path.join(RXHANDBD_DIR, "Test_Labels.csv")
TRAIN_IMG_DIR   = os.path.join(RXHANDBD_DIR, "Train_Set/")
TEST_IMG_DIR    = os.path.join(RXHANDBD_DIR, "Test_Set/")
MPI_PATH        = "../linked/patients_master.csv"
OUTPUT_DIR      = "../linked/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

Paths OK


## 2. Load Data

In [2]:
train_labels = pd.read_csv(TRAIN_LABELS)
test_labels  = pd.read_csv(TEST_LABELS)
mpi          = pd.read_csv(MPI_PATH)

print(f"Train labels : {len(train_labels)} rows")
print(f"Test labels  : {len(test_labels)} rows")
print(f"MPI patients : {len(mpi)} rows")
print()
print("Train label columns:")
print(train_labels.columns.tolist())
print()
print("Train label sample:")
print(train_labels.head(5))
print()
print("Test label sample:")
print(test_labels.head(5))

Train labels : 4463 rows
Test labels  : 1115 rows
MPI patients : 1163 rows

Train label columns:
['Images', 'Text']

Train label sample:
      Images         Text
0  P1116.jpg       Dexter
1  P1117.jpg  Clavusef sw
2  P1118.jpg   m-lucas 10
3  P1119.jpg    Dophyllin
4  P1120.jpg      Delfian

Test label sample:
      Images      Text
0  P0001.jpg  Nexcital
1  P0002.jpg   Inderen
2  P0003.jpg   Indever
3  P0004.jpg    Losita
4  P0005.jpg  Rivotril


## 3. Combine Train and Test Labels

In [3]:
# Standardize column names
train_labels.columns = [c.strip() for c in train_labels.columns]
test_labels.columns  = [c.strip() for c in test_labels.columns]

print("Train columns:", train_labels.columns.tolist())
print("Test columns :", test_labels.columns.tolist())

Train columns: ['Images', 'Text']
Test columns : ['Images', 'Text']


In [4]:
# Rename columns to standard names: image_file, medicine_label
# Adjust these based on actual column names printed above
train_labels = train_labels.rename(columns={
    train_labels.columns[0]: "image_file",
    train_labels.columns[1]: "medicine_label"
})
test_labels = test_labels.rename(columns={
    test_labels.columns[0]: "image_file",
    test_labels.columns[1]: "medicine_label"
})

# Add split column to track origin
train_labels["split"] = "train"
test_labels["split"]  = "test"

# Combine
all_labels = pd.concat([train_labels, test_labels], ignore_index=True)
all_labels = all_labels.dropna(subset=["medicine_label"]).reset_index(drop=True)

print(f"Total prescription images : {len(all_labels)}")
print()
print("Sample medicine labels:")
print(all_labels["medicine_label"].value_counts().head(15))

Total prescription images : 5578

Sample medicine labels:
medicine_label
Tablet         146
Napa           114
Dexter         100
Omeprazole      87
Injection       78
Algin           69
Neuro-B         68
Diet            59
Nexe            54
Ceftriaxone     52
Nexum           48
Voltalin        47
nexe            47
Capsule         45
Rolac           43
Name: count, dtype: int64


## 4. Extract Clean Medication Names from MPI

In [5]:
# Extract first word of medication (the drug name, not dosage)
# e.g. 'Simvastatin 10 MG Oral Tablet' → 'Simvastatin'
mpi["med_name_clean"] = mpi["primary_medication"].str.split().str[0].str.lower()

print("Cleaned medication names (top 15):")
print(mpi["med_name_clean"].value_counts().head(15))
print()
print("Sample RxHandBD labels (lowercased):")
print(all_labels["medicine_label"].str.lower().value_counts().head(15))

Cleaned medication names (top 15):
med_name_clean
acetaminophen          183
simvastatin             88
amlodipine              72
lisinopril              72
hydrochlorothiazide     68
ibuprofen               61
amoxicillin             52
naproxen                37
penicillin              32
nda020800               25
24                      20
1                       19
120                     18
nda020503               17
60                      17
Name: count, dtype: int64

Sample RxHandBD labels (lowercased):
medicine_label
napa           150
tablet         146
omeprazole     104
nexe           101
dexter         100
diet            94
voltalin        81
algin           78
injection       78
neuro-b         69
emistat         67
ceftriaxone     56
xinc b          52
ceevit          51
nexum           50
Name: count, dtype: int64


## 5. Build Medication → Patient Pool Lookup

In [6]:
# Group patients by their cleaned medication name
med_patient_pools = mpi.groupby("med_name_clean")["patient_id"].apply(list).to_dict()
all_patient_ids   = mpi["patient_id"].tolist()
all_med_names     = list(med_patient_pools.keys())

print(f"Unique medication names in MPI : {len(all_med_names)}")
print(f"Total patients in pool         : {len(all_patient_ids)}")

Unique medication names in MPI : 81
Total patients in pool         : 1163


## 6. Fuzzy Match Each Image Label to a Medication

In [7]:
FUZZY_THRESHOLD = 70  # Minimum similarity score to consider a match

def find_patient_for_prescription(label):
    """
    Fuzzy match the prescription image label against MPI medication names.
    Returns (patient_id, match_type, matched_medication, score)
    """
    label_clean = str(label).lower().strip()

    best_score = 0
    best_med   = None

    for med_name in all_med_names:
        score = fuzz.partial_ratio(label_clean, med_name)
        if score > best_score:
            best_score = score
            best_med   = med_name

    if best_score >= FUZZY_THRESHOLD and best_med is not None:
        pool = med_patient_pools[best_med]
        return (
            random.choice(pool),
            "fuzzy_match",
            best_med,
            best_score
        )
    else:
        return (
            random.choice(all_patient_ids),
            "random_fallback",
            None,
            best_score
        )


print("Linking prescription images to patient IDs...")
print("(This may take a moment)")

results = all_labels["medicine_label"].apply(find_patient_for_prescription)

all_labels["patient_id"]          = results.apply(lambda x: x[0])
all_labels["match_type"]          = results.apply(lambda x: x[1])
all_labels["matched_medication"]  = results.apply(lambda x: x[2])
all_labels["match_score"]         = results.apply(lambda x: x[3])

print("Done.")
print()
print("Match type distribution:")
print(all_labels["match_type"].value_counts())

Linking prescription images to patient IDs...
(This may take a moment)
Done.

Match type distribution:
match_type
random_fallback    4029
fuzzy_match        1549
Name: count, dtype: int64


## 7. Reorder Columns

In [8]:
cols = [
    "patient_id", "match_type", "matched_medication", "match_score",
    "image_file", "medicine_label", "split"
]
all_labels = all_labels[cols]

print("Final columns:")
print(all_labels.columns.tolist())
print()
all_labels.head(5)

Final columns:
['patient_id', 'match_type', 'matched_medication', 'match_score', 'image_file', 'medicine_label', 'split']



,patient_id,match_type,matched_medication,match_score,image_file,medicine_label,split
0,98120f78-9add-614d-d23f-1a600e4426e9,fuzzy_match,abuse-deterrent,83,P1116.jpg,Dexter,train
1,db2b8604-f8ea-0b47-1b95-2cf9d553a104,random_fallback,NaN,50,P1117.jpg,Clavusef sw,train
2,c8ff4993-f646-91ea-f261-c8622dd1500a,fuzzy_match,1,100,P1118.jpg,m-lucas 10,train
3,1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,random_fallback,NaN,56,P1119.jpg,Dophyllin,train
4,41ba25b8-f5ca-3bce-c26f-64b5ce13e525,random_fallback,NaN,50,P1120.jpg,Delfian,train


## 8. Quality Check

In [9]:
print("=== Prescriptions Linking Quality Check ===")
print(f"Total prescription images   : {len(all_labels)}")
print(f"Unique patients assigned    : {all_labels['patient_id'].nunique()}")
print(f"Patients with 0 images      : {len(mpi) - all_labels['patient_id'].nunique()}")
print()
print("Match type distribution:")
print(all_labels["match_type"].value_counts())
print()
print("Match score stats (fuzzy matches only):")
fuzzy = all_labels[all_labels["match_type"] == "fuzzy_match"]
print(fuzzy["match_score"].describe())
print()
print("Images per patient (stats):")
print(all_labels.groupby("patient_id").size().describe())
print()
print("Top 10 matched medications:")
print(fuzzy["matched_medication"].value_counts().head(10))
print()
print("Train/Test split distribution:")
print(all_labels["split"].value_counts())

=== Prescriptions Linking Quality Check ===
Total prescription images   : 5578
Unique patients assigned    : 1150
Patients with 0 images      : 13

Match type distribution:
match_type
random_fallback    4029
fuzzy_match        1549
Name: count, dtype: int64

Match score stats (fuzzy matches only):
count    1549.000000
mean       79.453196
std         9.163807
min        71.000000
25%        75.000000
50%        75.000000
75%        83.000000
max       100.000000
Name: match_score, dtype: float64

Images per patient (stats):
count    1150.000000
mean        4.850435
std         4.375765
min         1.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        56.000000
dtype: float64

Top 10 matched medications:
matched_medication
naproxen           225
abuse-deterrent    197
donepezil          128
simvastatin         75
fexofenadine        63
1                   57
albuterol           54
cefuroxime          45
etonogestrel        42
vitamin             38
Name: cou

## 9. Save Output

In [10]:
output_path = os.path.join(OUTPUT_DIR, "prescriptions_linked.csv")
all_labels.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {all_labels.shape}")

Saved → ../linked/prescriptions_linked.csv
Shape  : (5578, 7)


In [11]:
# ── Fix 1: Remove bad matches ──────────────────────────────────────────────
BAD_MATCHES = ["abuse-deterrent", "1", "nda020800", "deterrent"]

bad_mask = all_labels["matched_medication"].isin(BAD_MATCHES)
print(f"Bad matches to reassign: {bad_mask.sum()}")

# Reassign bad matches to random
all_labels.loc[bad_mask, "patient_id"]         = [random.choice(all_patient_ids) for _ in range(bad_mask.sum())]
all_labels.loc[bad_mask, "match_type"]         = "random_fallback"
all_labels.loc[bad_mask, "matched_medication"] = None
all_labels.loc[bad_mask, "match_score"]        = 0

print()
print("Updated match type distribution:")
print(all_labels["match_type"].value_counts())
print()
print("Top 10 matched medications after fix:")
print(all_labels[all_labels["match_type"] == "fuzzy_match"]["matched_medication"].value_counts().head(10))

Bad matches to reassign: 254

Updated match type distribution:
match_type
random_fallback    4283
fuzzy_match        1295
Name: count, dtype: int64

Top 10 matched medications after fix:
matched_medication
naproxen        225
donepezil       128
simvastatin      75
fexofenadine     63
albuterol        54
cefuroxime       45
etonogestrel     42
vitamin          38
amoxicillin      37
liletta          33
Name: count, dtype: int64
